In [1]:
import numpy as np
import scanpy as sc
import pandas as pd
import liana as li
import gseapy as gp
from gseapy import Msigdb

import recon
import recon.data
import recon.explore

In [9]:
rna = sc.read_h5ad("../output/Pseudo_singlecell.h5ad")
#rna = rna[:, :2000].to_memory()

In [5]:
rna.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample_name', 'cell_type'], dtype='object')

In [3]:
df = rna.to_df()
print(df.head())

                         UBE2Q2P2   HMGB1P1  RNU12-2P  SSX9P     EZHIP  \
TCGA.3L.AA1B.01_B.cells  0.008457  0.074918  0.001042    0.0  0.000261   
TCGA.4N.A93T.01_B.cells  0.002319  0.074579  0.000263    0.0  0.000000   
TCGA.4T.AA8H.01_B.cells  0.006301  0.077031  0.001634    0.0  0.000000   
TCGA.5M.AAT4.01_B.cells  0.000000  0.000000  0.000000    0.0  0.000000   
TCGA.5M.AAT5.01_B.cells  0.000000  0.000000  0.000000    0.0  0.000000   

                           EFCAB8   SRP14P1   TRIM75P  SPATA31B1P  REXO1L6P  \
TCGA.3L.AA1B.01_B.cells  0.001302  0.001562  0.000261         0.0       0.0   
TCGA.4N.A93T.01_B.cells  0.001312  0.001574  0.000000         0.0       0.0   
TCGA.4T.AA8H.01_B.cells  0.001634  0.002042  0.000000         0.0       0.0   
TCGA.5M.AAT4.01_B.cells  0.000000  0.000000  0.000000         0.0       0.0   
TCGA.5M.AAT5.01_B.cells  0.000000  0.000000  0.000000         0.0       0.0   

                         ...  C17orf106  C17orf107  C17orf108  C17orf28  \
TCGA.

In [4]:
# Load pre-computed GRN (or generate with ReCoN - see Tutorial 4)
grn_path = "RNA_grn.csv"
grn = pd.read_csv(grn_path)
grn = grn.sort_values(by="weight", ascending=False)[:500_000]
grn["source"] = grn["source"].str.capitalize()
grn["source"] = grn["source"] + '_TF'
grn["target"] = grn["target"].str.capitalize()
grn.head(3)

,source,target,weight
0,Hivep3_TF,Linc02810,23.286292
1,Mtf1_TF,Arhgap29-as1,22.319459
2,Jun_TF,Sgip1,21.659802


In [ ]:
li.method.cellphonedb(rna, 
            # NOTE by default the resource uses HUMAN gene symbols
            expr_prop=0.00,
            use_raw=False,
            groupby="cell_type",
            verbose=True, key_added='cpdb_res')

Using resource `consensus`.
Using `.X`!
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
446 features of mat are empty, they will be removed.
2671 samples of mat are empty, they will be removed.
Converting `cell_type` to categorical!
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:266: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/home/marcelo.hurtado/anaconda3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
0.05 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 5920 samples and 1766 features


100%|██████████| 1000/1000 [00:44<00:00, 22.57it/s]


In [ ]:
# Load receptor-gene links from NicheNet prior knowledge
receptor_genes = recon.data.load_data.load_receptor_genes("mouse_receptor_gene_from_NichenetPKN")
# for human, use "human_receptor_gene_from_NichenetPKN"

# Filter to genes present in our GRN
genes = np.unique(grn['source'].tolist() + grn['target'].tolist())
receptor_genes = receptor_genes[receptor_genes['target'].isin(genes)]
receptor_genes.head()

In [8]:
ccc_network = rna.uns["cpdb_res"].copy()
ccc_network = ccc_network[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_network = ccc_network.rename(columns={
    "lr_means": "weight",
    "source": "celltype_source",
    "target": "celltype_target",
    "ligand": "source",
    "receptor": "target"
})
ccc_network = ccc_network[ccc_network['weight'] != 0]

In [9]:
print(ccc_network.head(5))

     source target    weight celltype_source celltype_target
1457   Mrc1  Ptprc  3.505000      Macrophage            cDC2
1537   Mrc1  Ptprc  3.455000            cDC2            cDC2
1473   Mrc1  Ptprc  3.327027        Monocyte            cDC2
1553   Mrc1  Ptprc  3.305000             pDC            cDC2
497    Mrc1  Ptprc  2.705000      Macrophage        Monocyte
